# HAM10000 - Clasificador de Lesiones Cutáneas con ResNet50
**Dataset:** HAM10000 (Human Against Machine with 10000 training images)  
**Modelo:** ResNet50 preentrenado con ImageNet + Fine-tuning  
**Framework:** TensorFlow / Keras

## Clases del Dataset
| Abreviatura | Descripción |
|---|---|
| `akiec` | Queratosis actínica / carcinoma intraepitelial |
| `bcc` | Carcinoma basocelular |
| `bkl` | Lesiones benignas tipo queratosis |
| `df` | Dermatofibroma |
| `mel` | Melanoma |
| `nv` | Nevos melanocíticos |
| `vasc` | Lesiones vasculares |


**Estructura esperada del dataset:**
```
HAM10000/
├── HAM10000_images_part_1/   ← imágenes .jpg
├── HAM10000_images_part_2/   ← imágenes .jpg
└── HAM10000_metadata.csv     ← metadatos con etiquetas
```

## 1. Instalación de Dependencias

In [ ]:
#!pip install tensorflow pandas numpy matplotlib scikit-learn seaborn Pillow tqdm -q

## 2. Importaciones

In [ ]:
import os
import random
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from glob import glob

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import plot_model

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

print(f'TensorFlow version: {tf.__version__}')
print(f'GPUs disponibles: {len(tf.config.list_physical_devices("GPU"))}')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 3. Configuración e Hiperparámetros

In [ ]:
# ─────────────────────────────────────────────────────────────────
# RUTAS DEL DATASET
# Descarga desde: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
# Descomprime y ajusta BASE_DIR a la carpeta raíz del dataset.
# ─────────────────────────────────────────────────────────────────
BASE_DIR        = './ham10000'
IMG_DIR_1       = os.path.join(BASE_DIR, 'HAM10000_images_part_1')
IMG_DIR_2       = os.path.join(BASE_DIR, 'HAM10000_images_part_2')
METADATA_PATH   = os.path.join(BASE_DIR, 'HAM10000_metadata.csv')

# Hiperparámetros
IMG_SIZE         = (224, 224)
BATCH_SIZE       = 32
EPOCHS_FROZEN    = 15          # Fase 1: base congelada
EPOCHS_FINETUNE  = 25          # Fase 2: fine-tuning
LR_FROZEN        = 1e-3
LR_FINETUNE      = 1e-5
FINETUNE_FROM    = 140         # Descongelar desde esta capa (de ~175 en total)

# Clases
CLASS_NAMES  = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_LABELS = {
    'akiec': 'Queratosis Actínica',
    'bcc':   'Carcinoma Basocelular',
    'bkl':   'Queratosis Benigna',
    'df':    'Dermatofibroma',
    'mel':   'Melanoma',
    'nv':    'Nevo Melanocítico',
    'vasc':  'Lesión Vascular'
}
NUM_CLASSES = len(CLASS_NAMES)
print(f'Clases ({NUM_CLASSES}): {CLASS_NAMES}')

## 4. Carga y Exploración del Dataset

In [ ]:
df = pd.read_csv(METADATA_PATH)
print(f'Total de registros: {len(df)}')
print(f'Columnas: {df.columns.tolist()}')
df.head()

In [ ]:
# Construir mapa image_id → ruta de archivo
all_images = {}
for folder in [IMG_DIR_1, IMG_DIR_2]:
    for path in glob(os.path.join(folder, '*.jpg')):
        img_id = os.path.splitext(os.path.basename(path))[0]
        all_images[img_id] = path

print(f'Imágenes encontradas en disco: {len(all_images)}')
df['path'] = df['image_id'].map(all_images)
df = df.dropna(subset=['path'])
print(f'Imágenes con ruta válida:      {len(df)}')

In [ ]:
# Distribución de clases
class_counts = df['dx'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, NUM_CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

bars = axes[0].bar([CLASS_LABELS[c] for c in class_counts.index],
                   class_counts.values, color=colors)
axes[0].set_title('Distribución de Clases', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Número de muestras')
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, class_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(val), ha='center', fontweight='bold')

axes[1].pie(class_counts.values,
            labels=[CLASS_LABELS[c] for c in class_counts.index],
            autopct='%1.1f%%', colors=colors, startangle=140)
axes[1].set_title('Proporción de Clases', fontsize=14, fontweight='bold')

plt.tight_layout()
#plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(class_counts)

In [ ]:
# Visualizar 5 ejemplos por clase
fig, axes = plt.subplots(NUM_CLASSES, 5, figsize=(15, NUM_CLASSES * 2.5))
fig.suptitle('Muestras por clase', fontsize=15, fontweight='bold', y=1.01)

for i, cls in enumerate(CLASS_NAMES):
    samples = df[df['dx'] == cls].sample(5, random_state=SEED)
    for j, (_, row) in enumerate(samples.iterrows()):
        axes[i, j].imshow(Image.open(row['path']))
        axes[i, j].axis('off')
        if j == 0:
            axes[i, j].set_ylabel(CLASS_LABELS[cls], fontsize=9, fontweight='bold')

plt.tight_layout()
#plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. División del Dataset y Generadores de Datos

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# 1. Creamos las columnas de etiquetas numéricas y en string que el modelo necesita
df['label']     = df['dx'].map({cls: i for i, cls in enumerate(CLASS_NAMES)})
df['label_str'] = df['dx']

# 2. Separar un 15% para TEST, agrupando por la lesión (evitando el data leakage)
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trainval_idx, test_idx = next(gss_test.split(df, groups=df['lesion_id']))

df_trainval = df.iloc[trainval_idx].copy()
df_test = df.iloc[test_idx].copy()

# 3. Separar un 15% (del total original) para VALIDACIÓN, desde el grupo trainval
# El test_size relativo es aprox 0.1765
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED)
train_idx, val_idx = next(gss_val.split(df_trainval, groups=df_trainval['lesion_id']))

df_train = df_trainval.iloc[train_idx].copy()
df_val = df_trainval.iloc[val_idx].copy()

# 4. Imprimimos el resultado para comprobar que todo cuadra
print(f"Train:      {len(df_train):>5}  ({len(df_train)/len(df)*100:.1f}%)")
print(f"Validación: {len(df_val):>5}  ({len(df_val)/len(df)*100:.1f}%)")
print(f"Test:       {len(df_test):>5}  ({len(df_test)/len(df)*100:.1f}%)")

In [ ]:
# Pesos de clase para manejar el desbalance
cw_array = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=df_train['label'].values
)
class_weights = {i: w for i, w in enumerate(cw_array)}

print('Pesos por clase:')
for i, cls in enumerate(CLASS_NAMES):
    print(f'  [{i}] {CLASS_LABELS[cls]:<30} → {class_weights[i]:.4f}')

In [ ]:
preprocess_input = tf.keras.applications.resnet50.preprocess_input

# Generador de entrenamiento con data augmentation
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Sin augmentation para validación y test
val_test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

def make_generator(datagen, df_split, shuffle=False):
    return datagen.flow_from_dataframe(
        dataframe=df_split,
        x_col='path',
        y_col='label_str',
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        classes=CLASS_NAMES,
        shuffle=shuffle,
        seed=SEED
    )

train_gen = make_generator(train_datagen, df_train, shuffle=True)
val_gen   = make_generator(val_test_datagen, df_val)
test_gen  = make_generator(val_test_datagen, df_test)

print(f'Steps/epoch train: {len(train_gen)}')
print(f'Steps/epoch val:   {len(val_gen)}')
print(f'Steps test:        {len(test_gen)}')

## 6. Construcción del Modelo ResNet50

In [ ]:
def build_model(num_classes, input_shape=(224, 224, 3), freeze_base=True):
    """
    ResNet50 preentrenada con ImageNet + cabeza de clasificación personalizada.
    
    Arquitectura:
      Input → ResNet50 (base) → GlobalAvgPool → BN → Dense(512,relu)
              → Dropout(0.5) → Dense(256,relu) → Dropout(0.3) → Dense(N,softmax)
    """
    base = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    base.trainable = not freeze_base

    inputs = keras.Input(shape=input_shape, name='input_image')
    x = base(inputs, training=False)   # BN de la base en modo inferencia
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.BatchNormalization(name='bn_top')(x)
    x = layers.Dense(512, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.5, name='drop1')(x)
    x = layers.Dense(256, activation='relu', name='fc2')(x)
    x = layers.Dropout(0.3, name='drop2')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    return Model(inputs, outputs, name='HAM10000_ResNet50'), base


model, base_model = build_model(NUM_CLASSES, input_shape=IMG_SIZE + (3,), freeze_base=True)

model.compile(
    optimizer=Adam(learning_rate=LR_FROZEN),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

print(f'Capas base ResNet50:        {len(base_model.layers)}')
print(f'Parámetros totales:         {model.count_params():,}')
print(f'Parámetros entrenables:     {sum(np.prod(v.shape) for v in model.trainable_variables):,}')
model.summary()
plot_model(model, show_shapes=True, dpi=60, show_layer_names=True, rankdir='TB', to_file='modelo_resnet50.png')

## 7. Fase 1 - Entrenamiento con Base Congelada

In [ ]:
callbacks_p1 = [
    ModelCheckpoint('best_phase1.keras', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=7,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                      min_lr=1e-7, verbose=1)
]

print('=' * 55)
print('FASE 1: Base ResNet50 CONGELADA')
print('=' * 55)

history1 = model.fit(
    train_gen,
    epochs=EPOCHS_FROZEN,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_p1,
    verbose=1
)

## 8. Fase 2 - Fine-Tuning

In [ ]:
print('=' * 55)
print(f'FASE 2: Descongelando capas > {FINETUNE_FROM} de ResNet50')
print('=' * 55)

base_model.trainable = True

# Congelar las capas inferiores
for layer in base_model.layers[:FINETUNE_FROM]:
    layer.trainable = False

# BatchNorm siempre en modo inferencia para evitar inestabilidad
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print(f'Parámetros entrenables: {sum(np.prod(v.shape) for v in model.trainable_variables):,}')

model.compile(
    optimizer=Adam(learning_rate=LR_FINETUNE),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

callbacks_p2 = [
    ModelCheckpoint('best_finetune.keras', monitor='val_auc', mode='max',
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor='val_auc', mode='max', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4,
                      min_lr=1e-8, verbose=1)
]

history2 = model.fit(
    train_gen,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_p2,
    verbose=1
)

## 9. Visualización del Entrenamiento

In [ ]:
def plot_history(h1, h2, metrics=('loss', 'accuracy', 'auc')):
    combined = {k: h1.history[k] + h2.history[k] for k in h1.history}
    boundary = len(h1.history['loss'])
    x = range(1, len(combined['loss']) + 1)

    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 5))
    fig.suptitle('Historial de Entrenamiento', fontsize=14, fontweight='bold')

    for ax, m in zip(axes, metrics):
        ax.plot(x, combined[m],           'b-o', ms=3, label=f'Train {m}')
        ax.plot(x, combined[f'val_{m}'],  'r-o', ms=3, label=f'Val {m}')
        ax.axvline(boundary, color='green', ls='--', lw=1.5, label='Fine-Tuning →')
        ax.set_xlabel('Epoch'); ax.set_ylabel(m.capitalize())
        ax.set_title(m.capitalize()); ax.legend(); ax.grid(True, alpha=0.3)

    plt.tight_layout()
    #plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_history(history1, history2)

## 10. Evaluación en el Conjunto de Test

In [ ]:
best_model = keras.models.load_model('best_finetune.keras')

print('Evaluando en test...')
test_results = best_model.evaluate(test_gen, verbose=1)

print('\n' + '='*50)
print('MÉTRICAS EN TEST')
print('='*50)
for name, val in zip(best_model.metrics_names, test_results):
    print(f'  {name:<15}: {val:.4f}')

In [ ]:
# Predicciones
test_gen.reset()
y_proba = best_model.predict(test_gen, verbose=1)
y_pred  = np.argmax(y_proba, axis=1)
y_true  = test_gen.classes

print('\nREPORTE DE CLASIFICACIÓN')
print('='*70)
print(classification_report(
    y_true, y_pred,
    target_names=[CLASS_LABELS[c] for c in CLASS_NAMES]
))

In [ ]:
# Matrices de confusión
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, data, fmt, title in zip(
        axes,
        [cm, cm_norm],
        ['d', '.2f'],
        ['Conteos absolutos', 'Normalizada']):
    sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues' if fmt=='d' else 'YlOrRd',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5)
    ax.set_title(f'Matriz de Confusión - {title}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Real'); ax.set_xlabel('Predicho')

plt.tight_layout()
#plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Curvas ROC One-vs-Rest
y_bin = label_binarize(y_true, classes=range(NUM_CLASSES))
colors_roc = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

fig, ax = plt.subplots(figsize=(10, 8))
auc_scores = {}
for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
    roc_auc = auc(fpr, tpr)
    auc_scores[cls] = roc_auc
    ax.plot(fpr, tpr, color=col, lw=2,
            label=f'{CLASS_LABELS[cls]} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1],'k--',lw=1, label='Clasificador Aleatorio')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Curvas ROC - Multiclase (OvR)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAUC por clase:')
for cls, s in auc_scores.items():
    print(f'  {CLASS_LABELS[cls]:<35} AUC = {s:.4f}')
print(f'\n  Macro-AUC: {np.mean(list(auc_scores.values())):.4f}')

In [ ]:
# Visualización de predicciones individuales
test_df_reset = df_test.reset_index(drop=True)
idx = np.random.RandomState(SEED).choice(len(test_df_reset), 16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle('Predicciones - Conjunto de Test', fontsize=14, fontweight='bold')

for ax, i in zip(axes.flatten(), idx):
    img = Image.open(test_df_reset.loc[i, 'path']).resize(IMG_SIZE)
    true_cls = CLASS_NAMES[y_true[i]]
    pred_cls = CLASS_NAMES[y_pred[i]]
    conf     = y_proba[i, y_pred[i]]
    correct  = y_true[i] == y_pred[i]
    color    = '#27AE60' if correct else '#E74C3C'
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Real: {true_cls}\nPred: {pred_cls} ({conf:.1%})',
                 fontsize=8, color=color, fontweight='bold')

plt.tight_layout()
#plt.savefig('predictions_sample.png', dpi=120, bbox_inches='tight')
plt.show()

## 11. Guardar el Modelo Final

In [ ]:
#best_model.save('HAM10000_ResNet50_final.keras')
#best_model.save_weights('HAM10000_ResNet50.weights.h5')
#print('Modelo guardado: HAM10000_ResNet50_final.keras')
#print('Pesos guardados: HAM10000_ResNet50.weights.h5')

print('\n' + '='*55)
print('RESUMEN FINAL')
print('='*55)
print(f'  Epochs Fase 1:   {len(history1.history["loss"])}')
print(f'  Epochs Fase 2:   {len(history2.history["loss"])}')
print(f'  Mejor val_auc P1: {max(history1.history["val_auc"]):.4f}')
print(f'  Mejor val_auc P2: {max(history2.history["val_auc"]):.4f}')
for name, val in zip(best_model.metrics_names, test_results):
    print(f'  Test {name:<12}: {val:.4f}')
print(f'  Macro-AUC OvR:   {np.mean(list(auc_scores.values())):.4f}')

## 12. Función de Inferencia para Nuevas Imágenes

In [ ]:
def predict_image(image_path, model=best_model):
    """Predice la clase de una imagen de lesión cutánea y muestra la distribución."""
    prep = tf.keras.applications.resnet50.preprocess_input
    img = Image.open(image_path).convert('RGB').resize(IMG_SIZE)
    arr = prep(np.array(img, dtype=np.float32))
    arr = np.expand_dims(arr, 0)

    proba    = model.predict(arr, verbose=0)[0]
    pred_idx = int(np.argmax(proba))
    pred_cls = CLASS_NAMES[pred_idx]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.imshow(img); ax1.axis('off')
    ax1.set_title(f'Pred: {CLASS_LABELS[pred_cls]}\nConf: {proba[pred_idx]:.1%}',
                  fontweight='bold', fontsize=12)

    sorted_pairs = sorted(zip(CLASS_NAMES, proba), key=lambda x: x[1], reverse=True)
    lbls  = [CLASS_LABELS[c] for c, _ in sorted_pairs]
    vals  = [v for _, v in sorted_pairs]
    cols  = ['#E74C3C' if c == pred_cls else '#3498DB' for c, _ in sorted_pairs]
    bars  = ax2.barh(lbls, vals, color=cols)
    for bar, v in zip(bars, vals):
        ax2.text(v + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{v:.1%}', va='center', fontsize=9)
    ax2.set_xlim(0, 1.1)
    ax2.set_title('Probabilidades por clase', fontweight='bold')
    plt.tight_layout(); plt.show()

    return {'class': pred_cls, 'label': CLASS_LABELS[pred_cls],
            'confidence': float(proba[pred_idx]),
            'probabilities': dict(zip(CLASS_NAMES, proba.tolist()))}


# Ejemplo con una imagen aleatoria del test
sample_path = df_test.sample(1, random_state=0)['path'].values[0]
result = predict_image(sample_path)
print(f"\nClase predicha:  {result['class']} - {result['label']}")
print(f"Confianza:       {result['confidence']:.1%}")